# RAG 파이프라인 (Retrieval-Augmented Generation)

세 가지 공정 지식 소스를 벡터 DB에 저장하고, EXAONE LLM으로 질의응답을 수행하는 파이프라인입니다.

**파이프라인 순서:**
1. 문서 로드 (논문 rule JSON + OPLS 공정 JSON + SHAP 분석 Markdown)
2. 청크 분할 → 임베딩 (multilingual-e5-large-instruct) → Chroma DB 저장
3. LLM 로드 (EXAONE-3.5-7.8B-Instruct, 4bit 양자화)
4. RAG Chain 구성 및 질의응답

**필요 파일:**
- `../data/rag_data_all.json` — 논문 기반 공정 rule
- `../data/opls_process_knowledge.json` — 현업 OPLS 공정 조치 정보
- `../data/shap_analysis_for_rag.md` — SHAP 기반 모델 해석 정보

## 0. 환경 설정

In [1]:
#1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

#2. Github에서 코드 가져오기
!git clone https://github.com/heyitsmialee/PRAGma.git
%cd PRAGma

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'PRAGma' already exists and is not an empty directory.
/content/PRAGma


In [2]:
# EXAONE-3.5는 transformers>=4.50.0의 AttentionInterface.get_interface() API 필요
!pip install -q \
  "langchain==0.2.17" \
  "langchain-core==0.2.43" \
  "langchain-community==0.2.19" \
  "langchain-text-splitters==0.2.4" \
  "langchain-huggingface==0.0.3" \
  "langchain-chroma==0.1.4" \
  "chromadb>=0.6.0,<0.7.0"

!pip install -q "transformers==4.51.3" accelerate bitsandbytes

!pip install -q sentence-transformers torch
!pip install -q "ragas>=0.1.0,<0.2.0" datasets


# 설치 후 런타임 재시작이 필요할 수 있습니다.
# Runtime > Restart session 후 이 셀부터 다시 실행하세요.

ERROR: Cannot install chromadb<0.7.0 and >=0.6.0 and langchain-chroma==0.1.4 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


## 1. 문서 로드 함수

세 가지 소스를 각각 로드합니다.

- `load_json_as_documents`: JSON 항목 전체를 문자열로 변환하여 `Document` 생성, `type` 메타데이터 부여
- `load_markdown_as_documents`: TextLoader로 Markdown 파일 로드, `type=shap_analysis` 메타데이터 부여

In [3]:
import os
import json
from typing import List

import torch

from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from transformers import BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def load_json_as_documents(json_path: str, knowledge_type: str) -> List[Document]:
    documents = []

    if not os.path.exists(json_path):
        print(f"JSON 파일 없음: {json_path}")
        return documents

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for item in data:
        content = json.dumps(item, ensure_ascii=False, indent=2)

        documents.append(
            Document(
                page_content=content,
                metadata={
                    "source": json_path,
                    "type": knowledge_type,
                    "id": item.get("paper_id", item.get("opls_id", ""))
                }
            )
        )

    return documents


def load_markdown_as_documents(md_path: str) -> List[Document]:
    if not os.path.exists(md_path):
        print(f"Markdown 파일 없음: {md_path}")
        return []

    loader = TextLoader(file_path=md_path, encoding="utf-8")
    docs = loader.load()

    for doc in docs:
        doc.metadata["source"] = md_path
        doc.metadata["type"] = "shap_analysis"

    return docs

## 2. RAG 파이프라인 설정 함수

- **문서 로드**: `rag_data_all.json` (논문 rule) + `opls_process_knowledge.json` (OPLS 공정) + `shap_analysis_for_rag.md` (SHAP 분석)
- **청크 분할**: `chunk_size=700`, `chunk_overlap=100`
- **임베딩 모델**: `intfloat/multilingual-e5-large-instruct`
- **벡터 DB**: Chroma (`chroma_rag_data` 컬렉션), `rebuild_db=True`면 재구성 / `False`면 기존 DB 재사용
- **LLM**: `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct` (4bit 양자화)
- **Retriever**: 상위 `k=3` 청크 검색
- **프롬프트**: 핵심 답변 · 관련 공정 변수 · 근거 · 공정 조정 방향 포맷

In [4]:
from huggingface_hub import login
login(token="hf_xxx")

# 전역 캐시 초기화 — 이미 로드된 경우 재설정하지 않음 (셀 재실행 시 캐시 보존)
if '_embedding_model' not in globals():
    _embedding_model = None
if '_chat_llm' not in globals():
    _chat_llm = None


def load_models(force_reload=False):
    """임베딩 모델과 LLM을 최초 1회만 로드하고 전역 변수로 캐싱합니다.
    이미 로드된 경우 캐시를 재사용하므로 반복 실행해도 안전합니다.
    force_reload=True 로 호출하면 강제로 재로드합니다.
    """
    global _embedding_model, _chat_llm

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU를 찾을 수 없습니다.\n"
            "Runtime > Change runtime type > Hardware accelerator > GPU 로 변경 후 재실행하세요."
        )

    if _embedding_model is None or force_reload:
        print("임베딩 모델 로드 중...")
        _embedding_model = HuggingFaceEmbeddings(
            model_name="intfloat/multilingual-e5-large-instruct"
        )
        print("임베딩 모델 로드 완료")
    else:
        print("[캐시] 임베딩 모델 재사용")

    if _chat_llm is None or force_reload:
        print("EXAONE LLM 로드 중... (최초 1회, 수십 분 소요)")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        chat_model = HuggingFacePipeline.from_model_id(
            model_id="LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
            task="text-generation",
            pipeline_kwargs={
                "max_new_tokens": 1024,
                "do_sample": False,
                "repetition_penalty": 1.03
            },
            model_kwargs={
                "quantization_config": quantization_config,
                "trust_remote_code": True
            }
        )
        _chat_llm = ChatHuggingFace(llm=chat_model)
        print("LLM 로드 완료")
    else:
        print("[캐시] LLM 재사용")

    return _embedding_model, _chat_llm


def build_rag_chain(
    embedding_model,
    llm,
    paper_json_path="/content/drive/MyDrive/Colab Notebooks/PRAGma/rag_data_all.json",
    opls_json_path="/content/PRAGma/data/opls_process_knowledge.json",
    shap_md_path="/content/drive/MyDrive/Colab Notebooks/PRAGma/shap_analysis_for_rag.md",

    db_dir="./chroma_huggingface",
    rebuild_db=True
):
    """로드된 모델을 받아 문서 로드 → Chroma DB → RAG 체인을 구성합니다.
    모델 재로드 없이 DB·프롬프트를 바꿀 때 이 함수만 다시 호출하세요.
    Returns: (rag_chain, retriever)
    """
    print("1. 문서 로드 중...")
    document_list = []
    document_list.extend(load_json_as_documents(paper_json_path, "paper_rule"))
    document_list.extend(load_json_as_documents(opls_json_path, "opls_process_rule"))
    document_list.extend(load_markdown_as_documents(shap_md_path))

    if not document_list:
        print("로드된 문서가 없습니다.")
        return None, None

    print(f"로드된 문서 수: {len(document_list)}")

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
    split_docs = text_splitter.split_documents(document_list)
    print(f"분할된 Chunk 수: {len(split_docs)}")

    print("2. Chroma DB 설정 중...")
    if rebuild_db:
        database = Chroma.from_documents(
            documents=split_docs,
            embedding=embedding_model,
            collection_name="chroma_rag_data",
            persist_directory=db_dir
        )
    else:
        database = Chroma(
            collection_name="chroma_rag_data",
            embedding_function=embedding_model,
            persist_directory=db_dir
        )

    retriever = database.as_retriever(search_kwargs={"k": 3})

    template = """
다음 문맥을 참고하여 질문에 답변해 주세요.

문맥에는 논문 기반 공정 rule, 현업 OPLS 공정 조치 정보,
SHAP 기반 모델 해석 정보가 포함될 수 있습니다.

답변 시 아래 내용을 중심으로 정리해 주세요.
- 질문에 대한 핵심 답변
- 관련 공정 변수
- 모델 또는 문헌 기반 근거
- 필요 시 공정 조정 방향

문맥:
{context}

질문:
{question}

답변:
"""

    prompt = PromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(
            f"[source={doc.metadata.get('source', '')}, type={doc.metadata.get('type', '')}]\n{doc.page_content}"
            for doc in docs
        )

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("RAG Chain 구성 완료")
    return rag_chain, retriever

## 3. 질의응답 함수

`rag_chain.invoke(query)`로 질문 문자열을 직접 전달하고, 답변 문자열을 반환합니다.

In [5]:
def query_rag(rag_chain, query):
    if rag_chain is None:
        print("체인이 구성되지 않았습니다.")
        return None

    print(f"\n[질문]: {query}")

    answer = rag_chain.invoke(query)

    print("[답변]:\n")
    print(answer)

    return answer

## 4. 파이프라인 실행

두 단계로 나눠 실행합니다.

- **Step 1 — 모델 로드**: 최초 1회만 실행. 이미 로드된 경우 `[캐시]` 메시지 출력 후 즉시 반환.
- **Step 2 — RAG 체인 구성**: 모델을 재사용해 빠르게 재구성. 데이터나 파라미터를 바꿀 때 이 셀만 다시 실행하면 됩니다.

In [10]:
# Step 1: 모델 로드 (최초 1회만 실행 — 이미 로드된 경우 캐시 재사용)
embedding_model, llm = load_models()

[캐시] 임베딩 모델 재사용
EXAONE LLM 로드 중... (최초 1회, 수십 분 소요)


ValueError: Could not load the text-generation model due to missing dependencies.

In [ ]:
# Step 2: RAG 체인 구성 (모델 재로드 없이 빠르게 재구성 가능)
# 데이터나 파라미터를 바꿀 때 이 셀만 다시 실행하면 됩니다.
rag_chain, retriever = build_rag_chain(
    embedding_model=embedding_model,
    llm=llm,
    paper_json_path="/content/drive/MyDrive/Colab Notebooks/PRAGma/rag_data_all.json",
    opls_json_path="/content/drive/MyDrive/Colab Notebooks/PRAGma/opls_process_knowledge.json",
    shap_md_path="/content/drive/MyDrive/Colab Notebooks/PRAGma/shap_analysis_for_rag.md",
    db_dir="./chroma_huggingface",
    rebuild_db=True
)

## 5. 질의응답 테스트

In [ ]:
if rag_chain is not None:
    query = "Etching 온도와 비중이 높을 때 어떤 문제가 발생할 수 있고 어떻게 조치해야 하나요?"
    answer = query_rag(rag_chain, query)
else:
    print("RAG 파이프라인 구성 실패. 파일 경로를 확인해주세요.")

## 6. RAG 성능 평가

두 방법을 조합합니다.

- **수동 Q&A 셋**: 도메인 전문가가 작성한 5개 질문 + 기대 답변(ground truth)으로 정답 기반 평가
- **RAGAS 자동 평가**: 로드된 EXAONE 모델을 judge로 사용해 4가지 지표를 자동 계산

| 지표 | 설명 | ground truth 필요 |
|---|---|---|
| **Faithfulness** | 답변이 검색 문맥에 충실한가 (환각 탐지) | X |
| **Answer Relevancy** | 답변이 질문과 얼마나 관련 있는가 | X |
| **Context Precision** | 검색된 문맥이 답변 생성에 실제 기여했는가 | O |
| **Context Recall** | 정답에 필요한 정보가 검색된 문맥에 있었는가 | O |

In [ ]:
# 수동 Q&A 평가 데이터셋
# ground_truth는 도메인 전문가가 작성한 기대 답변입니다.
# 실제 데이터 내용에 맞게 수정하여 사용하세요.
QA_DATASET = [
    {
        "question": "Etching 온도가 높을 때 어떤 불량이 발생하며, 조치 방법은 무엇인가요?",
        "ground_truth": (
            "Etching 온도가 높으면 부식 속도가 과도하게 빨라져 선폭 감소, 과부식(over-etching), "
            "언더컷 불량이 발생합니다. 조치 방법은 온도를 정상 범위로 낮추거나 컨베이어 속도를 "
            "높여 접촉 시간을 줄이는 것입니다."
        ),
    },
    {
        "question": "Etching 비중이 높을 때 발생하는 문제와 조치는 무엇인가요?",
        "ground_truth": (
            "Etching 비중이 높으면 구리 이온 농도가 과포화 상태가 되어 부식 효율이 떨어지고 "
            "불균일한 에칭이 발생합니다. 순수(DIW)를 보충하거나 산화제를 추가하여 비중을 "
            "정상 범위로 낮춰야 합니다."
        ),
    },
    {
        "question": "SHAP 분석 결과에서 최종검사 수율에 가장 영향력이 높은 공정 변수는 무엇인가요?",
        "ground_truth": (
            "SHAP 분석 결과 Etching 온도, 비중, 컨베이어 속도가 최종검사 수율에 가장 큰 영향을 "
            "미치는 변수로 나타납니다. 이 변수들이 정상 범위를 벗어날 때 수율 저하와 "
            "상관관계가 높습니다."
        ),
    },
    {
        "question": "AOI 검사에서 이상이 감지되었을 때 우선 점검해야 할 공정 구간은 어디인가요?",
        "ground_truth": (
            "AOI 이상 감지 시 Etching 구간의 온도와 비중을 우선 확인합니다. "
            "이후 D/F(Dry Film) 노광·현상 조건, 도금 두께를 점검하며, "
            "OPLS 기준값과 현재 공정값을 비교하여 이탈 여부를 판단합니다."
        ),
    },
    {
        "question": "수율 예측 모델을 활용한 공정 이상 조기 감지 방법을 설명해주세요.",
        "ground_truth": (
            "수율 예측 모델은 실시간 공정 데이터를 입력받아 예측 수율이 임계값 이하로 "
            "떨어질 때 경보를 발생시킵니다. SHAP 값을 통해 어떤 변수가 수율 저하에 "
            "기여하는지 파악하고, OPLS 조치 기준에 따라 해당 공정 파라미터를 조정합니다."
        ),
    },
]

print(f"평가 데이터셋: {len(QA_DATASET)}개 Q&A 쌍 준비 완료")

In [ ]:
def collect_eval_data(qa_dataset, retriever, rag_chain):
    """각 질문에 대해 RAG 답변과 검색된 컨텍스트를 수집합니다."""
    eval_records = []
    total = len(qa_dataset)

    for i, item in enumerate(qa_dataset):
        question = item["question"]
        print(f"[{i+1}/{total}] {question[:45]}...")

        docs = retriever.invoke(question)
        contexts = [doc.page_content for doc in docs]
        answer = rag_chain.invoke(question)

        eval_records.append({
            "question": question,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": item["ground_truth"],
        })
        print(f"       답변 생성 완료 (컨텍스트 {len(contexts)}개 검색)")

    print(f"\n수집 완료: {len(eval_records)}개 레코드")
    return eval_records


# 평가 데이터 수집 실행
eval_records = collect_eval_data(QA_DATASET, retriever, rag_chain)

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
import pandas as pd

def run_ragas_evaluation(eval_records, llm, embedding_model):
    """RAGAS 4가지 지표로 RAG 파이프라인을 평가합니다.
    judge LLM으로 이미 로드된 EXAONE을 재사용합니다.
    """
    ragas_llm = LangchainLLMWrapper(llm)
    ragas_emb = LangchainEmbeddingsWrapper(embedding_model)

    metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
    for metric in metrics:
        metric.llm = ragas_llm
        if hasattr(metric, "embeddings"):
            metric.embeddings = ragas_emb

    dataset = Dataset.from_dict({
        "question":     [r["question"]     for r in eval_records],
        "answer":       [r["answer"]       for r in eval_records],
        "contexts":     [r["contexts"]     for r in eval_records],
        "ground_truth": [r["ground_truth"] for r in eval_records],
    })

    print("RAGAS 평가 실행 중... (질문당 LLM 호출 여러 번 발생)")
    result = evaluate(dataset=dataset, metrics=metrics)

    result_df = result.to_pandas()
    return result_df


result_df = run_ragas_evaluation(eval_records, llm, embedding_model)
print("\n===== RAGAS 평가 결과 =====")
print(result_df[["question", "faithfulness", "answer_relevancy",
                  "context_precision", "context_recall"]].to_string(index=False))
print(f"\n===== 평균 점수 =====")
print(result_df[["faithfulness", "answer_relevancy",
                  "context_precision", "context_recall"]].mean().round(4))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = "NanumGothic"   # 한글 폰트 (코랩 기본 제공)
matplotlib.rcParams["axes.unicode_minus"] = False

metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
metric_labels = ["Faithfulness\n(환각 방지)", "Answer\nRelevancy", "Context\nPrecision", "Context\nRecall"]
means = result_df[metric_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 왼쪽: 지표별 평균 점수 바 차트 ---
bars = axes[0].bar(metric_labels, means, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"], width=0.5)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Score (0~1)")
axes[0].set_title("RAGAS 평균 점수")
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.3f}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")

# --- 오른쪽: 질문별 히트맵 ---
heatmap_data = result_df[metric_cols].values
im = axes[1].imshow(heatmap_data, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
axes[1].set_xticks(range(len(metric_cols)))
axes[1].set_xticklabels(metric_labels, fontsize=9)
axes[1].set_yticks(range(len(result_df)))
axes[1].set_yticklabels([f"Q{i+1}" for i in range(len(result_df))], fontsize=9)
axes[1].set_title("질문별 점수 히트맵")
plt.colorbar(im, ax=axes[1])
for i in range(len(result_df)):
    for j in range(len(metric_cols)):
        axes[1].text(j, i, f"{heatmap_data[i, j]:.2f}", ha="center", va="center",
                     fontsize=9, color="black")

plt.tight_layout()
plt.savefig("ragas_evaluation_result.png", dpi=150, bbox_inches="tight")
plt.show()
print("결과 이미지 저장: ragas_evaluation_result.png")